<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/mobileNetV4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [2]:
import os
import shutil
import tensorflow as tf
import numpy as np
import tensorflow_hub as hub

from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [3]:
BASE = "/content/newdata"
IMG_SRC = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR = "/drive/MyDrive/checkpoints"
MODEL_PATH = "/drive/MyDrive/melanoma_mnv4_edge.keras"

In [4]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)

shutil.copytree(IMG_SRC, BASE)

'/content/newdata'

In [5]:
batch_size = 16
image_size = 224

In [6]:
def add_edge_map(image, label):

    image = tf.cast(image, tf.float32) / 255.0

    gray = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)

    edge = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))
    edge = edge / (tf.reduce_max(edge) + 1e-6)

    return (image, edge), label


def load_ds(path, shuffle):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(image_size, image_size),
        batch_size=batch_size,
        label_mode="categorical",
        shuffle=shuffle
    )

    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = load_ds(f"{BASE}/train", True)
val_ds = load_ds(f"{BASE}/valid", False)
test_ds = load_ds(f"{BASE}/test", False)

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [14]:
def get_mobilenetv4():

    url = "https://tfhub.dev/google/imagenet/mobilenet_v3_large_100_224/feature_vector/5"

    hub_layer = hub.KerasLayer(
        url,
        trainable=True
    )

    return hub_layer

In [15]:
def create_model():

    # ---------------- RGB INPUT ----------------
    rgb_input = layers.Input(shape=(image_size, image_size, 3))

    x = layers.RandomFlip("horizontal")(rgb_input)
    x = layers.RandomRotation(0.05)(x)

    x = layers.Rescaling(1./255)(x)

    backbone = get_mobilenetv4()
    feature_map = layers.Lambda(lambda t: backbone(t))(x)

    # =====================================================
    # EDGE BRANCH
    # =====================================================
    edge_input = layers.Input(shape=(image_size, image_size, 1))

    e = layers.Conv2D(32, 3, activation="relu", padding="same")(edge_input)
    e = layers.MaxPooling2D()(e)

    e = layers.Conv2D(64, 3, activation="relu", padding="same")(e)
    e = layers.MaxPooling2D()(e)

    e = layers.Conv2D(128, 3, activation="relu", padding="same")(e)

    # Reduce edge to vector (IMPORTANT FIX)
    e = layers.GlobalAveragePooling2D()(e)

    # =====================================================
    # FUSION
    # =====================================================
    fused = layers.Concatenate()([feature_map, e])

    fused = layers.Dense(256, activation="relu")(fused)
    fused = layers.BatchNormalization()(fused)
    fused = layers.Dropout(0.5)(fused)

    fused = layers.Dense(128, activation="relu")(fused)
    fused = layers.Dropout(0.3)(fused)

    outputs = layers.Dense(2, activation="softmax")(fused)

    model = tf.keras.Model(inputs=[rgb_input, edge_input], outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(3e-5),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"]
    )

    return model

In [16]:
model = create_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        320 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_flip_2       │ (None, 224, 224,  │          0 │ input_layer_2[0]… │
│ (RandomFlip)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ random_rotation_2   │ (None, 224, 224,  │          0 │ random_flip_2[0]… │
│ (RandomRotation)    │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ random_rotation_… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1280)      │          0 │ rescaling_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ conv2d_2[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1408)      │          0 │ lambda[0][0],     │
│ (Concatenate)       │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    360,704 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 487,554 (1.86 MB)

 Trainable params: 487,042 (1.86 MB)

 Non-trainable params: 512 (2.00 KB)

In [17]:
checkpoint = ModelCheckpoint(
    CHECKPOINT_DIR + "/best_mnv4_edge.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [18]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[checkpoint, early, lr]
)

Epoch 1/30
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step - accuracy: 0.7480 - loss: 0.5861
Epoch 1: val_accuracy improved from None to 0.66633, saving model to /drive/MyDrive/checkpoints/best_mnv4_edge.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_mnv4_edge.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 136s 244ms/step - accuracy: 0.8274 - loss: 0.4972 - val_accuracy: 0.6663 - val_loss: 0.6197 - learning_rate: 3.0000e-05
Epoch 2/30
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - accuracy: 0.8865 - loss: 0.3914
Epoch 2: val_accuracy improved from 0.66633 to 0.88911, saving model to /drive/MyDrive/checkpoints/best_mnv4_edge.keras

Epoch 2: finished saving model to /drive/MyDrive/checkpoints/best_mnv4_edge.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 125s 248ms/step - accuracy: 0.8832 - loss: 0.3892 - val_accuracy: 0.8891 - val_loss: 0.4348 - learning_rate: 3.0000e-05
Epoch 3/30
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.8890 - loss: 0.3666
Epoch 3: val_accuracy did not impro

In [19]:
loss, acc = model.evaluate(test_ds)

print("\nFINAL TEST ACCURACY:", acc)

model.save(MODEL_PATH)

63/63 ━━━━━━━━━━━━━━━━━━━━ 14s 219ms/step - accuracy: 0.8872 - loss: 0.3355

FINAL TEST ACCURACY: 0.8872255682945251
